In [1]:
import brightway2 as bw
from brightway2 import *
import time
import numpy as np
import openpyxl
import os
import pandas as pd
import glob

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
bw.projects.set_current('Prospective LCA v1') # set current project

In [4]:
databaseNames = bw.databases
myDatabaseNames = []
for databaseName in databaseNames:
    if "Abhi" in databaseName:
        myDatabaseNames.append(databaseName)

In [5]:
hydrogenResultsFileNames = ['hydrogen_GWP_results_distributed.xlsx', 
                           'hydrogen_humanHealth_results_distributed.xlsx', 
                           'hydrogen_ecosystems_results_distributed.xlsx', 
                           'hydrogen_resources_results_distributed.xlsx']

carbonDioxideResultsFileNames = ['carbon_dioxide_GWP_results_distributed.xlsx', 
                           'carbon_dioxide_humanHealth_results_distributed.xlsx', 
                           'carbon_dioxide_ecosystems_results_distributed.xlsx', 
                           'carbon_dioxide_resources_results_distributed.xlsx']

ammoniaResultsFileNames = ['ammonia_GWP_results_distributed.xlsx', 
                           'ammonia_humanHealth_results_distributed.xlsx', 
                           'ammonia_ecosystems_results_distributed.xlsx', 
                           'ammonia_resources_results_distributed.xlsx']

methanolResultsFileNames = ['methanol_GWP_results_distributed.xlsx', 
                           'methanol_humanHealth_results_distributed.xlsx', 
                           'methanol_ecosystems_results_distributed.xlsx', 
                           'methanol_resources_results_distributed.xlsx']

ethyleneResultsFileNames = ['ethylene_GWP_results_distributed.xlsx', 
                           'ethylene_humanHealth_results_distributed.xlsx', 
                           'ethylene_ecosystems_results_distributed.xlsx', 
                           'ethylene_resources_results_distributed.xlsx']

allResultsFileNames = hydrogenResultsFileNames + carbonDioxideResultsFileNames + ammoniaResultsFileNames + methanolResultsFileNames + ethyleneResultsFileNames

In [6]:
GWPMethod = [method for method in bw.methods if 'IPCC 2013' in str(method) and  'climate change' 
                in str(method) and 'GWP 100a' in str(method) and not 'no LT' in str(method)][0]
humanHealthMethod = [method for method in bw.methods if 'ReCiPe Endpoint (H,A)' in str(method)
                            and 'human health' in str(method) and 'total' in str(method) and
                            'w/o' not in str(method)][0]
ecosystemsMethod= [method for method in bw.methods if 'ReCiPe Endpoint (H,A)' in str(method)
                            and 'ecosystem' in str(method) and 'total' in str(method) and
                            'w/o' not in str(method)][0]
resourcesMethod= [method for method in bw.methods if 'ReCiPe Endpoint (H,A)' in str(method)
                        and 'resources' in str(method) and 'total' in str(method) and
                        'w/o' not in str(method)][0]                           

In [7]:
methodsList = [GWPMethod, humanHealthMethod, ecosystemsMethod, resourcesMethod]

In [8]:
def lca_calculations(activities, methodsList):
    results = np.zeros((len(activities), len(methodsList)))
    lca = LCA({activities[0] : 1}, method = methodsList[0]) # LCA object which will do all calculating
    lca.lci() # calculate inventory once to load all database data
    lca.decompose_technosphere() # keep the LU factorized matrices for faster calculations
    lca.lcia() # load method data
    characterizationMatrices = []
    
    for method in methodsList:
        lca.switch_method(method)
        characterizationMatrices.append(lca.characterization_matrix.copy())
        
    for first, activity in enumerate(activities):
        lca.redo_lci({activity : 1})
        
        for second, matrix in enumerate(characterizationMatrices):
            results[first, second] = (matrix * lca.inventory).sum()
            
    return results

In [9]:
def check_or_create_excel_file(filePath):
    if not os.path.exists(filePath):
        wb = openpyxl.Workbook()
        wb.save(filePath)
        print(f"Excel file created at {filePath}")

In [10]:
def delete_first_sheet_from_excel_files(filePath):
    wb = openpyxl.load_workbook(filePath)
    if wb.sheetnames[0] == 'Sheet':  # check if the workbook has any sheets
        firstSheet = wb.sheetnames[0]  # get the name of the first sheet
        wb.remove(wb[firstSheet])  # remove the first sheet
        wb.save(filePath)

In [11]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if "hydrogen, Abhi" in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(hydrogenResultsFileNames)):
        hydrogenResultsFilePath = os.path.join('..', 'Results', 'Hydrogen', hydrogenResultsFileNames[i])
        check_or_create_excel_file(hydrogenResultsFilePath)
        with pd.ExcelWriter(hydrogenResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\Hydrogen\hydrogen_GWP_results_distributed.xlsx
Excel file created at ..\Results\Hydrogen\hydrogen_humanHealth_results_distributed.xlsx
Excel file created at ..\Results\Hydrogen\hydrogen_ecosystems_results_distributed.xlsx
Excel file created at ..\Results\Hydrogen\hydrogen_resources_results_distributed.xlsx


In [12]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if "carbon dioxide, Abhi" in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(carbonDioxideResultsFileNames)):
        carbonDioxideResultsFilePath = os.path.join('..', 'Results', 'Carbon dioxide', carbonDioxideResultsFileNames[i])
        check_or_create_excel_file(carbonDioxideResultsFilePath)
        with pd.ExcelWriter(carbonDioxideResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\Carbon dioxide\carbon_dioxide_GWP_results_distributed.xlsx
Excel file created at ..\Results\Carbon dioxide\carbon_dioxide_humanHealth_results_distributed.xlsx
Excel file created at ..\Results\Carbon dioxide\carbon_dioxide_ecosystems_results_distributed.xlsx
Excel file created at ..\Results\Carbon dioxide\carbon_dioxide_resources_results_distributed.xlsx


In [13]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if "ammonia, Abhi" in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(ammoniaResultsFileNames)):
        ammoniaResultsFilePath = os.path.join('..', 'Results', 'Ammonia', ammoniaResultsFileNames[i])
        check_or_create_excel_file(ammoniaResultsFilePath)
        with pd.ExcelWriter(ammoniaResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\Ammonia\ammonia_GWP_results_distributed.xlsx
Excel file created at ..\Results\Ammonia\ammonia_humanHealth_results_distributed.xlsx
Excel file created at ..\Results\Ammonia\ammonia_ecosystems_results_distributed.xlsx
Excel file created at ..\Results\Ammonia\ammonia_resources_results_distributed.xlsx


In [14]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if "methanol, Abhi" in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(methanolResultsFileNames)):
        methanolResultsFilePath = os.path.join('..', 'Results', 'Methanol', methanolResultsFileNames[i])
        check_or_create_excel_file(methanolResultsFilePath)
        with pd.ExcelWriter(methanolResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\Methanol\methanol_GWP_results_distributed.xlsx
Excel file created at ..\Results\Methanol\methanol_humanHealth_results_distributed.xlsx
Excel file created at ..\Results\Methanol\methanol_ecosystems_results_distributed.xlsx
Excel file created at ..\Results\Methanol\methanol_resources_results_distributed.xlsx


In [15]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if "ethylene, Abhi" in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(ethyleneResultsFileNames)):
        ethyleneResultsFilePath = os.path.join('..', 'Results', 'Ethylene', ethyleneResultsFileNames[i])
        check_or_create_excel_file(ethyleneResultsFilePath)
        with pd.ExcelWriter(ethyleneResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\Ethylene\ethylene_GWP_results_distributed.xlsx
Excel file created at ..\Results\Ethylene\ethylene_humanHealth_results_distributed.xlsx
Excel file created at ..\Results\Ethylene\ethylene_ecosystems_results_distributed.xlsx
Excel file created at ..\Results\Ethylene\ethylene_resources_results_distributed.xlsx


In [16]:
allResultsFileNames = glob.glob(os.path.join('..', 'Results', '**/*.xls*'), recursive = True)
for filePath in allResultsFileNames:
    delete_first_sheet_from_excel_files(filePath)